# Free AI avatar studio: realistic mode (moving takes + LatentSync 1.5, Kaggle T4 x2)
Part of [free-ai-avatar-studio](https://github.com/sahilempire/free-ai-avatar-studio). For each scene of your kit it renders
moving **takes** with Wan2.1-Fun image-to-video (background motion, blinks, expressions) with the face **pinned to the
scene image at the first and last frame** (identity holds and takes chain without a seam), then dubs the scene's
voice-over onto them with LatentSync 1.5 (only the mouth is redrawn).

Same setup as the fast notebook: kit dataset as input, **GPU T4 x2**, Internet **On**. About 20 min per take plus
~1.2 min per second of speech for the dub. Download `takes.zip`, unzip into `work/takes/`, then `studio post` with
`studio track --source scenes`.

In [ ]:
# ---- Settings ----
TAKES_PER_SCENE = 2       # takes alternate A, B, A ... to cover each scene
SEEDS = [43, 71]
NEG = 'blurry, low quality, deformed face, distorted face, changing face, asymmetric eyes, distorted hands, extra fingers, fused fingers, melting phone, smeared objects, flicker, warping background, text, subtitles, watermark, logo'
DEFAULT_PROMPT = 'A person talks directly to the camera with natural lip movement, natural blinking, expressive eyebrows and subtle head movement. Realistic skin, natural light, hands steady.'

import os, glob, json, math, shutil, subprocess, threading, time, base64, re, torch
assert torch.cuda.is_available(), "Settings -> Accelerator -> GPU T4 x2"
print(torch.__version__, torch.cuda.device_count(), "x", torch.cuda.get_device_name(0))
hits = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert hits, "Add your kit as input: upload work/kit.zip (from `studio kit`) as a Kaggle dataset, then Add Input"
KIT = os.path.dirname(hits[0]); print("kit:", KIT)
def find(rel):  # Kaggle can flatten a dataset's folders, so fall back to looking files up by name
    hits = glob.glob(f"{KIT}/{rel}") or glob.glob(f"/kaggle/input/**/{os.path.basename(rel)}", recursive=True)
    assert hits, f"not in the dataset: {rel}"; return hits[0]
LOGS = "/kaggle/working/logs"; os.makedirs(LOGS, exist_ok=True); T0 = time.time()
def sh(cmd, log, cwd=None, env=None):
    t = time.time()
    with open(f"{LOGS}/{log}.log", "w") as f:
        r = subprocess.run(cmd, shell=True, cwd=cwd, env=env, stdout=f, stderr=subprocess.STDOUT)
    print(f"[{(time.time()-T0)/60:6.1f}m] {log}: exit {r.returncode} in {(time.time()-t)/60:.1f} min", flush=True)
    assert r.returncode == 0, f"{log} failed:\n" + "".join(open(f"{LOGS}/{log}.log").readlines()[-25:])
MEMPEAK, STOP = [0.0], threading.Event()
def mem_watch():  # system RAM log (two renderers at once can get OOM-killed)
    with open(f"{LOGS}/mem.log", "w") as f:
        while not STOP.is_set():
            m = {l.split(":")[0]: int(l.split()[1]) for l in open("/proc/meminfo")}
            used = (m["MemTotal"] - m["MemAvailable"]) / 2**20; MEMPEAK[0] = max(MEMPEAK[0], used)
            f.write(f"{(time.time()-T0)/60:6.1f}m used {used:5.1f} GiB\n"); f.flush(); STOP.wait(10)
threading.Thread(target=mem_watch, daemon=True).start()

SCENES = json.load(open(f"{KIT}/scenes.json"))
W = "/kaggle/working/library"
for d in (f"{W}/takes", "/kaggle/working/takes"): os.makedirs(d, exist_ok=True)
print([(s["id"], s["dur"]) for s in SCENES])

In [ ]:
# ---- Setup: LatentSync 1.5 (pinned packages kept apart) + VideoX-Fun + weights ----
LS, LSPKG, VF, WAN = "/tmp/LatentSync", "/tmp/ls_pkgs", "/tmp/VideoX-Fun", "/tmp/models/Wan2.1-Fun-V1.1-1.3B-InP"
if not os.path.exists(LS):
    sh(f"git clone -q https://github.com/bytedance/LatentSync.git {LS} && cd {LS} && git checkout -q a229c3948406bc2cf6eaf4873e662e70c6a04746", "ls_clone")
sh(f"pip install -q --no-deps --target {LSPKG} diffusers==0.32.2 transformers==4.48.0 'tokenizers>=0.21,<0.22' huggingface_hub==0.30.2 DeepCache==0.1.1", "ls_pins")
sh("pip install -q insightface==0.7.3 onnxruntime-gpu kornia decord ffmpeg-python omegaconf einops librosa soundfile", "ls_deps")
sh(f"python -c \"from huggingface_hub import hf_hub_download as d; [d('ByteDance/LatentSync-1.5', f, local_dir='{LS}/checkpoints') for f in ('latentsync_unet.pt','whisper/tiny.pt')]\"", "ls_weights")
if not os.path.exists(VF):
    sh(f"git clone -q https://github.com/aigc-apps/VideoX-Fun.git {VF} && cd {VF} && git checkout -q e8c314fbfa494d0429bbef605c9f6112b79a7ead", "vf_clone")
sh("pip install -q -U diffusers transformers accelerate ftfy timm tomesd torchdiffeq torchsde sentencepiece", "vf_deps")
sh(f"python -c \"from huggingface_hub import snapshot_download as s; s('alibaba-pai/Wan2.1-Fun-V1.1-1.3B-InP', local_dir='{WAN}')\"", "wan_weights")
BASE_SRC = open(f"{VF}/examples/wan2.1_fun/predict_i2v.py", encoding="utf-8").read()
CN_NEG = re.search(r'^negative_prompt\s*=\s*"(.*)"', BASE_SRC, re.M).group(1)
FREE_T5 = """
_encode = pipeline.encode_prompt
def _encode_then_free_t5(*a, **k):  # model_cpu_offload would keep T5 (11 GB) on the GPU through the VAE encode
    out = _encode(*a, **k); pipeline.text_encoder.to("cpu"); torch.cuda.empty_cache(); return out
pipeline.encode_prompt = _encode_then_free_t5
"""

In [ ]:
# ---- Takes, one after another on GPU 0 (two image-to-video runs at once nearly fill Kaggle's RAM) ----
def make_take(scene, k):
    out = f"{W}/takes/{scene['id']}_{k}.mp4"
    if os.path.exists(out): return out
    img = find(scene["image"]); prompt = scene.get("prompt") or DEFAULT_PROMPT
    SET = {"GPU_memory_mode": '"model_cpu_offload"', "model_name": repr(WAN), "sampler_name": '"Flow_Unipc"', "sample_size": "[864, 480]",
           "video_length": "81", "weight_dtype": "torch.float16", "validation_image_start": repr(img), "validation_image_end": repr(img),
           "prompt": repr(prompt), "negative_prompt": repr(NEG + ", " + CN_NEG), "num_inference_steps": "25",
           "seed": str(SEEDS[k % len(SEEDS)]), "save_path": repr(f"/tmp/wan_{scene['id']}_{k}")}
    src = BASE_SRC
    for key, v in SET.items():
        src, n = re.subn(rf"^{key}(\s*)=.*$", lambda m: f"{key}{m.group(1)}= {v}", src, count=1, flags=re.M); assert n == 1, key
    anchor = "coefficients = get_teacache_coefficients(model_name)"; assert src.count(anchor) == 1
    open(f"{VF}/examples/wan2.1_fun/predict_take.py", "w", encoding="utf-8").write(src.replace(anchor, FREE_T5 + anchor))
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    sh("python examples/wan2.1_fun/predict_take.py", f"take_{scene['id']}_{k}", cwd=VF, env=env)
    raw = sorted(glob.glob(f"/tmp/wan_{scene['id']}_{k}/*.mp4"))[-1]
    sh(f"ffmpeg -y -loglevel error -i {raw} -vf \"minterpolate=fps=25:mi_mode=mci:mc_mode=aobmc:me_mode=bidir:vsbmc=1\" -c:v libx264 -crf 16 -pix_fmt yuv420p {out}",
       f"interp_{scene['id']}_{k}")
    return out
for s in SCENES:
    for k in range(TAKES_PER_SCENE): make_take(s, k)

In [ ]:
# ---- Dub each scene: chain its takes to the scene length, LatentSync over the exact voice-over slice ----
for g in (0, 1):  # LatentSync writes ./temp, so each GPU worker gets its own copy (checkpoints shared via a symlink)
    if not os.path.exists(f"{LS}_{g}"):
        shutil.copytree(LS, f"{LS}_{g}", ignore=shutil.ignore_patterns("checkpoints")); os.symlink(f"{LS}/checkpoints", f"{LS}_{g}/checkpoints")
def dub(i, gpu):
    sc = SCENES[i]; out = f"/kaggle/working/takes/{sc['id']}.mp4"
    if os.path.exists(out): return
    wd = f"/tmp/scene_{sc['id']}"; os.makedirs(wd, exist_ok=True)
    with open(f"{wd}/chunks.txt", "w") as f:
        for c in sc["chunks"]: f.write(f"file '{find('audio/' + c + '.wav')}'\n")
    sh(f"ffmpeg -y -loglevel error -f concat -safe 0 -i {wd}/chunks.txt {wd}/vo.wav", f"vo_{sc['id']}")
    takes = [f"{W}/takes/{sc['id']}_{k}.mp4" for k in range(TAKES_PER_SCENE)]
    with open(f"{wd}/takes.txt", "w") as f:
        for j in range(int(sc["dur"] // 4.9) + 1): f.write(f"file '{takes[j % len(takes)]}'\n")
    sh(f"ffmpeg -y -loglevel error -f concat -safe 0 -i {wd}/takes.txt -t {sc['dur'] + 0.3:.3f} -c:v libx264 -crf 14 -pix_fmt yuv420p {wd}/drive.mp4", f"drive_{sc['id']}")
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONPATH=f"{LSPKG}:{LS}_{gpu}")
    sh(f"python -m scripts.inference --unet_config_path configs/unet/stage2.yaml --inference_ckpt_path checkpoints/latentsync_unet.pt "
       f"--inference_steps 25 --guidance_scale 1.5 --enable_deepcache --video_path {wd}/drive.mp4 --audio_path {wd}/vo.wav --video_out_path {out}",
       f"dub_{sc['id']}", cwd=f"{LS}_{gpu}", env=env)
def worker(gpu, ids):
    for i in ids:
        try: dub(i, gpu)
        except AssertionError as e: print(f"{SCENES[i]['id']} failed on gpu{gpu}: {str(e)[:300]}")
order = sorted(range(len(SCENES)), key=lambda i: -SCENES[i]["dur"])
ts = [threading.Thread(target=worker, args=(g, order[g::2])) for g in (0, 1)]
for t in ts: t.start()
for t in ts: t.join()
STOP.set(); shutil.make_archive("/kaggle/working/takes", "zip", "/kaggle/working/takes")
missing = [s["id"] for s in SCENES if not os.path.exists(f"/kaggle/working/takes/{s['id']}.mp4")]
print(f"peak RAM {MEMPEAK[0]:.1f} GiB | {(time.time()-T0)/60:.1f} min | download /kaggle/working/takes.zip")
assert not missing, f"scenes missing: {missing} (see logs); run again to finish them"